In [1]:
import zipfile
import os

zip_path = "1.zip"
extract_path = "unzipped_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully to:", extract_path)

Unzipped successfully to: unzipped_data


In [2]:
import os
import re
import cv2
import pandas as pd
import easyocr

BASE_DIR = "unzipped_data/1"
OUTPUT_CSV = "image_code_output.csv"

reader = easyocr.Reader(['en'], gpu=False)

def clean_text(text):
    return (text
            .replace("O", "0")
            .replace("o", "0")
            .replace("I", "1")
            .replace("l", "1")
            .replace("S", "5")
            .replace("B", "8"))

def extract_code(text):
    matches = re.findall(r"\d{6,9}", text)
    for m in matches:
        if 7 <= len(m) <= 8:
            return m
    return matches[0] if matches else None

Using CPU. Note: This module is much faster with a GPU.


In [ ]:
def process_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    h, w = img.shape[:2]

    crop = img[int(h * 0.7):h, 0:w]

    result = reader.readtext(crop)
    text = " ".join([r[1] for r in result])
    text = clean_text(text)

    code = extract_code(text)

    if not code:
        result = reader.readtext(img)
        text = " ".join([r[1] for r in result])
        text = clean_text(text)
        code = extract_code(text)

    return code

In [ ]:
rows = []

for file in os.listdir(BASE_DIR):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.normpath(os.path.join(BASE_DIR, file)).replace('\\', '/')

        try:
            code = process_image(image_path)
            print(code, image_path)

            rows.append({
                "code": code,
                "image_path": image_path
            })

        except Exception as e:
            print(f"Error: {image_path} -> {e}")
            rows.append({
                "code": None,
                "image_path": image_path
            })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("\nDone! CSV saved at:", OUTPUT_CSV)

Using CPU. Note: This module is much faster with a GPU.


271002935 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.22.57 PM.jpeg
10028269 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.22.58 PM (1).jpeg
10028268 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.22.58 PM.jpeg
10025542 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.22.59 PM (1).jpeg
10027276 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.22.59 PM.jpeg
10027632 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.00 PM (1).jpeg
None unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.00 PM.jpeg
100294 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.01 PM (1).jpeg
10029460 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.01 PM (2).jpeg
None unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.01 PM.jpeg
10029028 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.02 PM (1).jpeg
10029129 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.02 PM.jpeg
10029029 unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.03 PM.jpeg
None unzipped_data/1/WhatsApp Image 2026-04-29 at 1.23.04 PM (1).jpeg
10029